In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    build_feature_combos,
    detect_device,
    dropna_splits,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data


In [2]:
DATA_SET = 'rand_C'


df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

OUTPUT_DIR = make_run_dir()

## GPU auto-detect


In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
BATCH_SIZE = CFG['BATCH']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  batch=65,536  |  dtype=torch.bfloat16


## Hyperparameters


In [ ]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'LR={INIT_LR:.4f} (scaled from {BASE_LR} at batch {BASE_BATCH})  '
      f'BATCH={BATCH_SIZE:,}  WARMUP={WARMUP_EPOCHS} epochs')


MAX_EPOCHS=100  PATIENCE=30  LR=0.0160 (scaled from 0.001 at batch 4096)  BATCH=65,536  WARMUP=5 epochs


## Feature definitions


In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag','vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

feature_combos = build_feature_combos(FEATURES_3, EXTRA_FEATURES, max_extra=3)

n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')


Total feature combinations: 130
  Base (3F):  1
  3F + 1:     9
  3F + 2:     36
  3F + 3:     84


## Drop rows with NaN and pre-allocate on GPU


In [6]:
required_cols = list(set(ALL_FEATURES + [TARGET]))
df_train, df_val, df_test, nan_stats = dropna_splits(df_train, df_val, df_test, required_cols)
print(f'Rows before: {nan_stats["before"]:,}  after: {nan_stats["after"]:,}  dropped: {nan_stats["dropped"]:,}')

gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Rows before: 2,452,545  after: 2,400,097  dropped: 52,448
Data on GPU  |  VRAM used: 0.59 GB / 85 GB  |  Free: 84.5 GB
Train: 1,680,100  Val: 479,984  Test: 240,013  Features: 12


## Analytic benchmark


In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 62.1649  RMSE = 0.016094
Coefficients: a = -0.079595, b = -0.092957, c = -0.082286


## Train all feature combinations


In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

print(f'\n{"=" * 70}')
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(f'{"=" * 70}\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 130 MODELS
  GPU: A100-80GB  |  Batch: 65,536  |  AMP: True  |  Max epochs: 100

  [  1/130] 3F                             SSE=59.0414  Gain=+5.0%  ep=100  14.8s  elapsed=0.2min
  [ 25/130] 3F+iv_lag+rho                  SSE=38.7791  Gain=+37.6%  ep=100  8.1s  elapsed=3.5min
  [ 50/130] 3F+vix_lag+iv_lag+gamma        SSE=26.7012  Gain=+57.0%  ep=100  8.3s  elapsed=6.9min
  [ 75/130] 3F+iv_lag+d_iv_lag+vix_mom_lag SSE=26.4356  Gain=+57.5%  ep=100  8.3s  elapsed=10.3min
  [100/130] 3F+d_iv_lag+vix_mom_lag+rho    SSE=28.1707  Gain=+54.7%  ep=100  8.2s  elapsed=13.8min
  [125/130] 3F+vix_mom+theta+rho           SSE=45.9912  Gain=+26.0%  ep=100  8.3s  elapsed=17.2min
  [130/130] 3F+theta+vega+rho              SSE=53.1540  Gain=+14.5%  ep=100  8.3s  elapsed=17.9min

Done: 17.9 min for 130 models (avg 8.2s/model)


## Results summary


In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,62.164894,0.000259,0.016094,0.005848,0.001250,0.002617,0.070734,None,None,None
1,3F,59.041412,0.000246,0.015684,0.007230,-0.000264,0.004366,0.117425,9.1s,5.02%,None
2,3F+vix_lag,59.976768,0.000250,0.015808,0.008270,-0.000324,0.005400,0.103443,8.2s,3.52%,-1.58%
3,3F+iv_lag,40.164711,0.000167,0.012936,0.007615,-0.000161,0.005045,0.399601,8.1s,35.39%,33.03%
4,3F+d_iv_lag,35.195595,0.000147,0.012110,0.007065,0.000180,0.004583,0.473882,8.0s,43.38%,12.37%
...,...,...,...,...,...,...,...,...,...,...,...
127,3F+gamma+theta+vega,50.732944,0.000211,0.014539,0.007797,0.000028,0.005080,0.241623,8.3s,18.39%,-0.35%
128,3F+gamma+theta+rho,50.858242,0.000212,0.014557,0.007885,0.000652,0.005221,0.239750,8.4s,18.19%,-0.25%
129,3F+gamma+vega+rho,59.071014,0.000246,0.015688,0.008217,-0.001012,0.005523,0.116982,8.1s,4.98%,-16.15%
130,3F+theta+vega+rho,53.153973,0.000221,0.014882,0.007897,0.000964,0.005184,0.205433,8.3s,14.50%,10.02%


## Top 10 by Gain vs Hull-White


In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+iv_lag+gamma+theta,6,23.0686,0.009804,62.89,8.4
1,3F+iv_lag+d_iv_lag+vix_mom,6,23.9712,0.009994,61.44,8.2
2,3F+iv_lag+d_iv_lag+theta,6,26.1839,0.010445,57.88,8.2
3,3F+iv_lag+d_iv_lag+vix_mom_lag,6,26.4356,0.010495,57.48,8.3
4,3F+vix_lag+iv_lag+theta,6,26.4634,0.010500,57.43,8.2
5,3F+vix_lag+iv_lag+gamma,6,26.7012,0.010547,57.05,8.3
6,3F+vix_lag+iv_lag+vix_mom,6,27.0698,0.010620,56.45,8.3
7,3F+vix_lag+iv_lag+d_iv_lag,6,27.1203,0.010630,56.37,8.1
8,3F+vix_lag+d_iv_lag+theta,6,27.2682,0.010659,56.14,8.1
9,3F+d_iv_lag+vix_mom_lag,5,27.4130,0.010687,55.90,8.2


## Best per group (3F, +1, +2, +3)


In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f'{nf_label}: {best["combo_name"]}')
    print(f'    SSE={best["SSE"]:.4f}  RMSE={best["RMSE"]:.6f}  Gain={best["Gain_vs_HW_%"]:.2f}%\n')

3F (base): 3F
    SSE=59.0414  RMSE=0.015684  Gain=5.02%

+1 (4F): 3F+d_iv_lag
    SSE=35.1956  RMSE=0.012110  Gain=43.38%

+2 (5F): 3F+d_iv_lag+vix_mom_lag
    SSE=27.4130  RMSE=0.010687  Gain=55.90%

+3 (6F): 3F+iv_lag+gamma+theta
    SSE=23.0686  RMSE=0.009804  Gain=62.89%



## Summary statistics


In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 17.7 min (0.30 hr)
Models trained: 130
Best overall: 3F+iv_lag+gamma+theta (Gain=62.89%)


## Cleanup


In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.73 GB / 85 GB
Total training time: 17.7 min for 130 models
